# Pipeline de clima — versão go horse

Tudo num notebook só: URL escrita na mão, variáveis globais, nenhuma função, e copia e cola quando aparece um caso novo.

**Funciona.** O problema aparece quando alguém muda alguma coisa. Compare com [`02_com_facade.ipynb`](02_com_facade.ipynb).

In [1]:
import requests
import pandas as pd
import sqlite3

## Extração: um request por cidade, coordenadas coladas no código

In [2]:
dfs = []

r = requests.get("https://archive-api.open-meteo.com/v1/archive?latitude=-22.91&longitude=-43.17&start_date=2026-01-01&end_date=2026-03-31&daily=temperature_2m_max,temperature_2m_min,precipitation_sum&timezone=America/Sao_Paulo")
d = r.json()["daily"]
df1 = pd.DataFrame(d)
df1["cidade"] = "Rio de Janeiro"
dfs.append(df1)

r = requests.get("https://archive-api.open-meteo.com/v1/archive?latitude=-23.55&longitude=-46.63&start_date=2026-01-01&end_date=2026-03-31&daily=temperature_2m_max,temperature_2m_min,precipitation_sum&timezone=America/Sao_Paulo")
d = r.json()["daily"]
df2 = pd.DataFrame(d)
df2["cidade"] = "São Paulo"
dfs.append(df2)

r = requests.get("https://archive-api.open-meteo.com/v1/archive?latitude=-19.92&longitude=-43.94&start_date=2026-01-01&end_date=2026-03-31&daily=temperature_2m_max,temperature_2m_min,precipitation_sum&timezone=America/Sao_Paulo")
d = r.json()["daily"]
df3 = pd.DataFrame(d)
df3["cidade"] = "Belo Horizonte"
dfs.append(df3)

df = pd.concat(dfs)
df.head()

,time,temperature_2m_max,temperature_2m_min,precipitation_sum,cidade
0,2026-01-01,33.4,25.4,0.1,Rio de Janeiro
1,2026-01-02,34.8,25.8,3.6,Rio de Janeiro
2,2026-01-03,31.5,25.2,1.6,Rio de Janeiro
3,2026-01-04,26.8,21.8,9.0,Rio de Janeiro
4,2026-01-05,25.7,21.5,4.4,Rio de Janeiro


## Limpeza: o mínimo para não dar erro

In [3]:
df = df.rename(columns={"time": "data", "temperature_2m_max": "tmax", "temperature_2m_min": "tmin", "precipitation_sum": "chuva_mm"})
df["data"] = pd.to_datetime(df["data"])
df = df.dropna()
df["mes"] = df["data"].dt.strftime("%Y-%m")

## Agregação

In [4]:
mensal = df.groupby(["cidade", "mes"]).agg(
    tmax_media=("tmax", "mean"),
    tmin_media=("tmin", "mean"),
    chuva_total_mm=("chuva_mm", "sum"),
).round(1).reset_index()
mensal

,cidade,mes,tmax_media,tmin_media,chuva_total_mm
0,Belo Horizonte,2026-01,25.8,18.3,432.8
1,Belo Horizonte,2026-02,26.3,18.8,378.5
2,Belo Horizonte,2026-03,26.9,18.3,164.1
3,Rio de Janeiro,2026-01,30.6,23.4,142.9
4,Rio de Janeiro,2026-02,30.5,24.3,212.4
5,Rio de Janeiro,2026-03,28.6,22.8,85.6
6,São Paulo,2026-01,26.9,18.3,282.0
7,São Paulo,2026-02,27.2,19.3,280.9
8,São Paulo,2026-03,27.5,18.0,171.1


## Carga: `append` no banco (rodou duas vezes, duplicou)

In [5]:
con = sqlite3.connect("clima.db")
df.to_sql("clima_diario", con, if_exists="append", index=False)
mensal.to_sql("clima_mensal", con, if_exists="append", index=False)
print(pd.read_sql("SELECT COUNT(*) AS linhas FROM clima_diario", con))
con.close()

   linhas
0     270


## Relatório

In [6]:
texto = "RELATORIO CLIMA\n\n"
for i, row in mensal.iterrows():
    texto += f"{row['cidade']} {row['mes']}: max {row['tmax_media']} min {row['tmin_media']} chuva {row['chuva_total_mm']}\n"
open("relatorio.txt", "w").write(texto)
print(texto)

RELATORIO CLIMA

Belo Horizonte 2026-01: max 25.8 min 18.3 chuva 432.8
Belo Horizonte 2026-02: max 26.3 min 18.8 chuva 378.5
Belo Horizonte 2026-03: max 26.9 min 18.3 chuva 164.1
Rio de Janeiro 2026-01: max 30.6 min 23.4 chuva 142.9
Rio de Janeiro 2026-02: max 30.5 min 24.3 chuva 212.4
Rio de Janeiro 2026-03: max 28.6 min 22.8 chuva 85.6
São Paulo 2026-01: max 26.9 min 18.3 chuva 282.0
São Paulo 2026-02: max 27.2 min 19.3 chuva 280.9
São Paulo 2026-03: max 27.5 min 18.0 chuva 171.1



---

## Chegou um caso novo: rodar com o CSV que veio de outra fonte

Sem tempo, copiamos as células de cima e trocamos a leitura. O arquivo `clima_sujo.csv` tem defeitos típicos de dado real: nulos, um sensor marcando 999 °C, chuva negativa, `tmin` e `tmax` trocados e linhas repetidas.

In [7]:
df = pd.read_csv("../dados/clima_sujo.csv")
df["data"] = pd.to_datetime(df["data"])
df = df.dropna()
df["mes"] = df["data"].dt.strftime("%Y-%m")

mensal = df.groupby(["cidade", "mes"]).agg(
    tmax_media=("tmax", "mean"),
    tmin_media=("tmin", "mean"),
    chuva_total_mm=("chuva_mm", "sum"),
).round(1).reset_index()
mensal

,cidade,mes,tmax_media,tmin_media,chuva_total_mm
0,Belo Horizonte,2026-01,25.7,18.4,432.8
1,Belo Horizonte,2026-02,26.3,18.8,378.5
2,Belo Horizonte,2026-03,26.9,18.3,164.1
3,Rio de Janeiro,2026-01,31.1,23.5,142.1
4,Rio de Janeiro,2026-02,30.5,24.3,207.1
5,Rio de Janeiro,2026-03,28.6,22.8,85.6
6,São Paulo,2026-01,59.2,18.2,281.6
7,São Paulo,2026-02,27.2,19.3,280.9
8,São Paulo,2026-03,27.5,18.0,168.0


Repare em **São Paulo, janeiro**: a máxima média foi para 59,2 °C porque o 999 °C entrou na conta (o certo é 26,9 °C). No **Rio, fevereiro**, o `dropna` jogou fora um dia inteiro por causa de um único campo vazio, e a chuva do mês caiu de 212,4 para 207,1 mm. As linhas repetidas entram duas vezes nas médias. Ninguém percebe, porque não dá erro.

## Pontos de dor (para a apresentação)

1. **Acoplamento:** quem usa o pipeline precisa saber URL, parâmetros, nomes de colunas da API e a ordem dos passos.
2. **Duplicação:** cada caso novo é uma cópia. Um bug corrigido numa cópia continua na outra.
3. **Troca de fonte ou destino:** mudar de API para CSV ou de SQLite para Parquet exige mexer em todas as cópias.
4. **Sem validação:** dado ruim passa calado e o resultado sai errado.
5. **Estado global:** `df` e `mensal` são sobrescritos. A ordem em que as células rodam muda o resultado.
6. **Não dá para testar:** não existe uma função para chamar no teste.